<a href="https://colab.research.google.com/github/annatsamoyra-prog/data-story-/blob/main/newsit_gr_texnologia_2026_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q requests beautifulsoup4 pandas numpy


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re


In [3]:
main_url = "https://www.newsit.gr"
category_url = "https://www.newsit.gr/category/texnologia"

start_page = 1
end_page = 40   # ανώτατο όριο ασφαλείας -- η λούπα θα σταματήσει μόνη της νωρίτερα (auto-stop)

TARGET_YEAR = 2026

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "el-GR,el;q=0.9,en-US;q=0.8,en;q=0.7",
}

# Ένα άρθρο τεχνολογίας είναι πάντα /texnologia/<slug>/<αριθμός>/
ARTICLE_PATH_RE = re.compile(r"/texnologia/[^/]+/\d+/?$")

# Ημερομηνία teaser στη σελίδα κατηγορίας: "23:04  09.09.26" (HH:MM, δύο κενά, DD.MM.YY)
TEASER_DATE_RE = re.compile(r"(\d{1,2}):(\d{2})\s+(\d{1,2})\.(\d{1,2})\.(\d{2})\b")

# Boilerplate ενότητες μέσα στο άρθρο -- μόλις τις συναντήσουμε σταματάμε να μαζεύουμε κείμενο
# (επιβεβαιωμένο από πραγματικό άρθρο).
STOP_HEADINGS = [
    "Σχόλια",
    "Περισσότερα σχόλια",
    "Αν τα χάσατε",
    "Τεχνολογία: Περισσότερα άρθρα",
    "Τεχνολογία: όλες οι ειδήσεις",
]

# Γραμμές που παραλείπουμε αλλά ΔΕΝ σταματάμε (μπορεί να εμφανιστούν νωρίς)
JUNK_LINE_PREFIXES = [
    "Ακολουθήστε το",
    "Σχολίασε εδώ",
    "Προσθήκη του newsit.gr",
]


## ΒΗΜΑ 3: Μαζεύουμε τα urls των άρθρων (teasers) -- με auto-stop στο 2026

Σε κάθε σελίδα κατηγορίας διαβάζουμε την ορατή ημερομηνία δίπλα σε κάθε τίτλο. Αν μια σελίδα
έχει *όλες* τις ημερομηνίες της πριν το 2026, σταματάμε -- δεν χρειάζεται να συνεχίσουμε πιο πίσω.


In [4]:
teasers_list = []

for page_num in range(start_page, end_page + 1):
    page_url = category_url if page_num == 1 else f"{category_url}/page/{page_num}/"
    print(f"Σελίδα {page_num}: {page_url}")

    response = requests.get(page_url, headers=HEADERS)
    doc = BeautifulSoup(response.text, "html.parser")

    #*** teaser_blocks -- κάθε άρθρο-teaser σε ξεχωριστό block (WordPress <article> tags) ***
    teaser_blocks = doc.find_all("article")
    if not teaser_blocks:
        # fallback αν το θέμα δεν χρησιμοποιεί <article> tags για τα teasers
        teaser_blocks = doc.find_all("div", class_=re.compile(r"post|article|item", re.IGNORECASE))

    if not teaser_blocks:
        print("  Κανένα teaser block βρέθηκε -- σταματάω.")
        break

    page_years = []

    for block in teaser_blocks:
        link_tag = None
        for a in block.find_all("a", href=True):
            if ARTICLE_PATH_RE.search(a["href"]):
                link_tag = a
                break
        if link_tag is None:
            continue

        href = link_tag["href"]
        url = href if href.startswith("http") else main_url + href
        story_dict = {"url": url}

        block_text = block.get_text(" ", strip=True)
        m = TEASER_DATE_RE.search(block_text)
        if m:
            yy = int(m.group(5))
            full_year = 2000 + yy
            page_years.append(full_year)

        teasers_list.append(story_dict)

    time.sleep(1)

    # Auto-stop: αν βρέθηκαν ημερομηνίες σε αυτή τη σελίδα και ΟΛΕΣ είναι πριν το 2026,
    # δεν χρειάζεται να συνεχίσουμε πιο πίσω.
    if page_years and max(page_years) < TARGET_YEAR:
        print(f"  Σελίδα {page_num}: όλες οι ημερομηνίες είναι πριν το {TARGET_YEAR} -- σταματάω.")
        break

newsit_teasers_df = pd.DataFrame(teasers_list)
newsit_teasers_df = newsit_teasers_df.drop_duplicates(subset=["url"]).reset_index(drop=True)
print("\nΣυνολικά teasers:", len(newsit_teasers_df))
newsit_teasers_df


Σελίδα 1: https://www.newsit.gr/category/texnologia
Σελίδα 2: https://www.newsit.gr/category/texnologia/page/2/
Σελίδα 3: https://www.newsit.gr/category/texnologia/page/3/
  Σελίδα 3: όλες οι ημερομηνίες είναι πριν το 2026 -- σταματάω.

Συνολικά teasers: 49


,url
0,https://www.newsit.gr/texnologia/pos-to-3d-pri...
1,https://www.newsit.gr/texnologia/h-apple-parou...
2,https://www.newsit.gr/texnologia/apple-simera-...
3,https://www.newsit.gr/texnologia/ta-tria-pio-p...
4,https://www.newsit.gr/texnologia/apple-to-neo-...
5,https://www.newsit.gr/texnologia/cosmote-telek...
6,https://www.newsit.gr/texnologia/anafores-gia-...
7,https://www.newsit.gr/texnologia/epese-to-face...
8,https://www.newsit.gr/texnologia/apple-nea-epo...
9,https://www.newsit.gr/texnologia/i-texniti-noi...


## ΒΗΜΑ 4: Δοκιμή σε ένα άρθρο

In [5]:
article_url = newsit_teasers_df.loc[0, "url"]
print(article_url)

response = requests.get(article_url, headers=HEADERS)
doc = BeautifulSoup(response.text, "html.parser")

# Το <article> είναι το τυπικό container ενός WordPress single-post template.
# Αν δεν βρεθεί, πέφτουμε πίσω σε όλη τη σελίδα.
article = doc.find("article")
if article is None:
    article = doc

print(article.prettify()[:3000])


https://www.newsit.gr/texnologia/pos-to-3d-printing-allazei-ti-viomixania-kai-tin-kathimerinotita-mas/4771552/
<article class="post-4771552 post type-post status-publish format-standard has-post-thumbnail category-texnologia no-featured-image-padding" id="post-4771552">
 <section class="post-media mb-2">
  <div class="featured-image-container expand-image-lt-sm rel">
   <picture class="featured-image-container article-image fw fh rel" style="aspect-ratio: 16 / 9;">
    <div class="video-wrapper column center-center">
     <iframe allowfullscreen="" encrypted-media;="" frameborder="0" height="100%" picture-in-picture"="" src="https://www.youtube.com/embed/fjSIiEh7ECA?" type="video/mp4" width="100%" wmode="transparent">
     </iframe>
    </div>
   </picture>
  </div>
 </section>
 <div class="share-inside-the-content mb-2 row space-between-center gap-sm">
  <div class="share-menu comments parent-share-menu rel row start-center fs-base bg-color">
   <a class="comments-big-bubble rel text-

In [6]:
# Τίτλος -- προτιμάμε το og:title meta (πάντα καθαρό), αλλιώς το h1
og_title = doc.find("meta", {"property": "og:title"})
title = og_title["content"] if og_title else (article.find("h1").text if article.find("h1") else None)
title


'Πώς το 3D printing αλλάζει τη βιομηχανία και την καθημερινότητά μας'

In [7]:
# Ημερομηνία -- meta property="article:published_time", ISO format με timezone (UTC),
# έτοιμο για pd.to_datetime χωρίς επεξεργασία.
meta_date = doc.find("meta", {"property": "article:published_time"})
date = meta_date["content"] if meta_date else None
date


'2026-09-11T07:26:41+00:00'

In [8]:
# Συντάκτης -- meta name="author"
meta_author = doc.find("meta", {"name": "author"})
author = meta_author["content"] if meta_author else None
author


'Χρυσάνθη Παππά'

In [9]:
# Πλήρες κείμενο -- μαζεύουμε p/h2/h3 μέχρι να συναντήσουμε boilerplate ενότητα.
p_texts_list = []
for el in article.find_all(["p", "h2", "h3"]):
    text = el.get_text(" ", strip=True)
    if not text:
        continue
    if text in STOP_HEADINGS or any(text.startswith(h) for h in STOP_HEADINGS):
        break
    if any(text.startswith(prefix) for prefix in JUNK_LINE_PREFIXES):
        continue
    p_texts_list.append(text)

full_text = " ".join(p_texts_list)
full_text = "".join(full_text.splitlines())
full_text


'Η τρισδιάστατη εκτύπωση δεν αποτελεί πλέον τεχνολογία του μέλλοντος. Είναι ήδη μέρος της σύγχρονης παραγωγής, της βιομηχανίας, της έρευνας, της ιατρικής και της ανάπτυξης νέων προϊόντων, αλλάζοντας τον τρόπο με τον οποίο σχεδιάζονται και κατασκευάζονται αντικείμενα. Στο νέο βίντεο του newsit.gr επισκεφθήκαμε τις εγκαταστάσεις της 3DHUB Greece και συνομιλήσαμε με τον Γιάννη Αθανασίου , 3D Solutions Engineer, ο οποίος εξηγεί πώς λειτουργεί στην πράξη το οικοσύστημα των τεχνολογιών 3D και ποιες δυνατότητες προσφέρει σήμερα σε επιχειρήσεις, βιομηχανίες και επαγγελματίες. Από το 3D scanning και το reverse engineering μέχρι τον σχεδιασμό, το πρωτότυπο και την τελική παραγωγή, η διαδικασία μπορεί πλέον να ολοκληρωθεί με μεγάλη ακρίβεια και ταχύτητα, αξιοποιώντας διαφορετικές τεχνολογίες τρισδιάστατης εκτύπωσης ανάλογα με τις ανάγκες κάθε εφαρμογής. Στη συνέντευξη παρουσιάζονται οι σημαντικότερες εφαρμογές του 3D printing, οι διαφορές ανάμεσα στις διαθέσιμες τεχνολογίες, καθώς και ο ρόλος που

## ΒΗΜΑ 5: Η λούπα για όλα τα άρθρα

In [10]:
full_articles_list = []

for article_url in newsit_teasers_df["url"]:
    response = requests.get(article_url, headers=HEADERS)
    doc = BeautifulSoup(response.text, "html.parser")

    article = doc.find("article")
    if article is None:
        article = doc

    full_article_dict = {}
    full_article_dict["site"] = "newsit.gr"
    full_article_dict["url"] = article_url

    try:
        og_title = doc.find("meta", {"property": "og:title"})
        title = og_title["content"] if og_title else article.find("h1").text
        full_article_dict["title"] = title
    except Exception:
        full_article_dict["title"] = None

    try:
        meta_date = doc.find("meta", {"property": "article:published_time"})
        full_article_dict["date"] = meta_date["content"] if meta_date else None
    except Exception:
        full_article_dict["date"] = None

    try:
        meta_author = doc.find("meta", {"name": "author"})
        full_article_dict["author"] = meta_author["content"] if meta_author else None
    except Exception:
        full_article_dict["author"] = None

    try:
        p_texts_list = []
        for el in article.find_all(["p", "h2", "h3"]):
            text = el.get_text(" ", strip=True)
            if not text:
                continue
            if text in STOP_HEADINGS or any(text.startswith(h) for h in STOP_HEADINGS):
                break
            if any(text.startswith(prefix) for prefix in JUNK_LINE_PREFIXES):
                continue
            p_texts_list.append(text)
        full_text = " ".join(p_texts_list)
        full_text = "".join(full_text.splitlines())
        full_article_dict["full_text"] = full_text if full_text else None
    except Exception:
        full_article_dict["full_text"] = None

    full_articles_list.append(full_article_dict)
    time.sleep(1)

newsit_full_articles_df = pd.DataFrame(full_articles_list)
newsit_full_articles_df = newsit_full_articles_df[["site", "url", "title", "date", "author", "full_text"]]
newsit_full_articles_df


,site,url,title,date,author,full_text
0,newsit.gr,https://www.newsit.gr/texnologia/pos-to-3d-pri...,Πώς το 3D printing αλλάζει τη βιομηχανία και τ...,2026-09-11T07:26:41+00:00,Χρυσάνθη Παππά,Η τρισδιάστατη εκτύπωση δεν αποτελεί πλέον τεχ...
1,newsit.gr,https://www.newsit.gr/texnologia/h-apple-parou...,H Apple παρουσίασε το πρώτο αναδιπλούμενο iPho...,2026-09-09T20:04:14+00:00,Βάσω Δελημήτρου,Η Apple έκανε το μεγαλύτερο σχεδιαστικό άλμα σ...
2,newsit.gr,https://www.newsit.gr/texnologia/apple-simera-...,Apple: Σήμερα τα αποκαλυπτήρια για το πρώτο αν...,2026-09-09T09:10:14+00:00,Μαργαρίτα Τζαγκαράκη,Ιστορικής σημασίας είναι η σημερινή ημέρα για ...
3,newsit.gr,https://www.newsit.gr/texnologia/ta-tria-pio-p...,Τα τρία πιο περίεργα πράγματα που μπορούν να χ...,2026-08-22T11:02:00+00:00,Χρύσα Πανούση,Και πόσοι δεν έχουν πληρώσει πιο ακριβά για να...
4,newsit.gr,https://www.newsit.gr/texnologia/apple-to-neo-...,Apple: Το νέο πρόγραμμα leasing για iPhone - Ο...,2026-07-30T16:48:08+00:00,Έρικα Τεκτονίδου,Το νέο πρόγραμμα Apple Upgrade την κορυφαίας ε...
5,newsit.gr,https://www.newsit.gr/texnologia/cosmote-telek...,COSMOTE TELEKOM: Το πρώτο δίκτυο κινητής στον ...,2026-07-28T08:17:39+00:00,Μαίρη Καλουτσάκη,Το δίκτυο κινητής της COSMOTE TELEKOM είναι το...
6,newsit.gr,https://www.newsit.gr/texnologia/anafores-gia-...,Αναφορές για προβλήματα σε Facebook και Messenger,2026-07-22T12:26:17+00:00,Μαργαρίτα Τζαγκαράκη,Προβλήματα αντιμετώπισαν το μεσημέρι της Τετάρ...
7,newsit.gr,https://www.newsit.gr/texnologia/epese-to-face...,Έπεσε το Facebook - Χιλιάδες αναφορές για προβ...,2026-07-19T07:57:03+00:00,Χρύσα Πανούση,Προβλήματα παρουσιάστηκαν το πρωί της Κυριακής...
8,newsit.gr,https://www.newsit.gr/texnologia/apple-nea-epo...,Apple: Νέα εποχή στα iPhone με τη νέα Apple In...,2026-06-09T10:09:05+00:00,Μαργαρίτα Τζαγκαράκη,Ένα μεγάλο event έκανε η Apple το βράδυ της Δε...
9,newsit.gr,https://www.newsit.gr/texnologia/i-texniti-noi...,Η Τεχνητή Νοημοσύνη απειλεί τους φυσικούς πόρο...,2026-06-04T09:24:43+00:00,Χρύσα Πανούση,Τεράστια η κατανάλωση νερού και ενέργειας που ...


## ΒΗΜΑ 6: Καθαρισμός -- κενές γραμμές & διπλότυπα

In [11]:
n_initial = len(newsit_full_articles_df)
print("Αρχικές εγγραφές (πριν τον καθαρισμό):", n_initial)


Αρχικές εγγραφές (πριν τον καθαρισμό): 49


In [12]:
nan_rows = newsit_full_articles_df[newsit_full_articles_df["full_text"].isna()]
print("Κενές γραμμές full_text:", len(nan_rows))

newsit_full_articles_df = newsit_full_articles_df.dropna(subset=["full_text"]).reset_index(drop=True)


Κενές γραμμές full_text: 4


In [13]:
duplicate_rows = newsit_full_articles_df[newsit_full_articles_df.duplicated(subset=["full_text"], keep=False)]
print("Διπλές εγγραφές:", len(duplicate_rows))

newsit_full_articles_df = newsit_full_articles_df.drop_duplicates(subset="full_text", keep="first").reset_index(drop=True)


Διπλές εγγραφές: 0


## ΒΗΜΑ 7: datetime + τελικό φίλτρο 2026

Το `date` μένει ως το αρχικό ISO string. Μετατρέπουμε σε πραγματικό datetime (με σωστή μετάφραση
στην ώρα Αθήνας, αφού η πηγή είναι UTC) σε **νέα** στήλη `datetime`, και μετά φιλτράρουμε στο 2026.


In [14]:
newsit_full_articles_df["datetime"] = pd.to_datetime(
    newsit_full_articles_df["date"], errors="coerce", utc=True
).dt.tz_convert("Europe/Athens")

newsit_2026_df = newsit_full_articles_df[newsit_full_articles_df["datetime"].dt.year == TARGET_YEAR].copy()
newsit_2026_df = newsit_2026_df.sort_values("datetime").reset_index(drop=True)
newsit_2026_df = newsit_2026_df[["site", "url", "title", "date", "author", "full_text", "datetime"]]

print("Πλήθος άρθρων 2026:", len(newsit_2026_df))
newsit_2026_df.head()


Πλήθος άρθρων 2026: 18


,site,url,title,date,author,full_text,datetime
0,newsit.gr,https://www.newsit.gr/texnologia/epanilthe-to-...,"Επανήλθε το X, χιλιάδες αναφορές για προβλήματα",2026-01-16T15:51:26+00:00,Μαργαρίτα Τζαγκαράκη,Προβλήματα στη λειτουργία του Χ (πρώην Twitter...,2026-01-16 17:51:26+02:00
1,newsit.gr,https://www.newsit.gr/texnologia/google-maps-e...,Google maps: Επικαιροποιούνται οι χάρτες - Πού...,2026-02-04T12:38:00+00:00,Κατερίνα Σερέτη,Οχήματα της Google θα κυκλοφορούν το επόμενο δ...,2026-02-04 14:38:00+02:00
2,newsit.gr,https://www.newsit.gr/texnologia/mat-slixt-o-d...,"Ματ Σλιχτ: Ο δημιουργός του Moltbook, του «soc...",2026-02-07T05:31:00+00:00,Μαργαρίτα Τζαγκαράκη,"Από σχολιαστής τεχνολογίας στα social media, δ...",2026-02-07 07:31:00+02:00
3,newsit.gr,https://www.newsit.gr/texnologia/apple-anakoin...,Apple: Ανακοίνωσε το φθηνότερο iPhone 17e και ...,2026-03-03T10:40:55+00:00,Τόνια Γκόρου,Η Apple παρουσίασε το νέο iPhone 17e και το iP...,2026-03-03 12:40:55+02:00
4,newsit.gr,https://www.newsit.gr/texnologia/neo-kakovoulo...,Νέο κακόβουλο λογισμικό που μπορεί να «κλέψει»...,2026-04-29T13:45:04+00:00,Σπύρος Πλουμίδης,Το Ερευνητικό Κέντρο της εταιρείας κυβερνοασφά...,2026-04-29 16:45:04+03:00


## Σύγκριση: πριν vs μετά τον καθαρισμό

In [17]:
n_final = len(newsit_2026_df)

print("Αρχικές εγγραφές (πριν τον καθαρισμό):", n_initial)
print("Τελικές εγγραφές (μετά τον καθαρισμό + φίλτρο 2026):", n_final)
print("Αφαιρέθηκαν:", n_initial - n_final)


Αρχικές εγγραφές (πριν τον καθαρισμό): 49
Τελικές εγγραφές (μετά τον καθαρισμό + φίλτρο 2026): 18
Αφαιρέθηκαν: 31


## Γρήγορος έλεγχος πριν το save

In [15]:
print("Σύνολο άρθρων:", len(newsit_2026_df))
print("\nMissing values ανά στήλη:")
print(newsit_2026_df.isna().sum())
print("\nΆρθρα ανά μήνα:")
print(newsit_2026_df["datetime"].dt.to_period("M").value_counts().sort_index())
print("\nMin datetime:", newsit_2026_df["datetime"].min())
print("Max datetime:", newsit_2026_df["datetime"].max())


Σύνολο άρθρων: 18

Missing values ανά στήλη:
site         0
url          0
title        0
date         0
author       0
full_text    0
datetime     0
dtype: int64

Άρθρα ανά μήνα:
datetime
2026-01    1
2026-02    2
2026-03    1
2026-04    1
2026-05    3
2026-06    2
2026-07    4
2026-08    1
2026-09    3
Freq: M, Name: count, dtype: int64

Min datetime: 2026-01-16 17:51:26+02:00
Max datetime: 2026-09-11 10:26:41+03:00


/tmp/ipykernel_1211/3922613284.py:5: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  print(newsit_2026_df["datetime"].dt.to_period("M").value_counts().sort_index())


## ΒΗΜΑ 8: Αποθήκευση στο GitHub

Χωρίς Drive -- απευθείας upload. Άλλαξε το `path` αν θες διαφορετικό όνομα αρχείου.


In [16]:
import base64
from google.colab import userdata

def save_df_to_github(df, repo, path, token=None, branch="main", message="Update dataset"):
    token = token or userdata.get("newtoken")
    csv_content = df.to_csv(index=False, encoding="utf-8-sig")
    content_b64 = base64.b64encode(csv_content.encode("utf-8-sig")).decode("utf-8")

    url = f"https://api.github.com/repos/{repo}/contents/{path}"
    headers = {"Authorization": f"token {token}", "Accept": "application/vnd.github+json"}

    existing = requests.get(url, headers=headers, params={"ref": branch})
    sha = existing.json().get("sha") if existing.status_code == 200 else None

    payload = {"message": message, "content": content_b64, "branch": branch}
    if sha:
        payload["sha"] = sha

    response = requests.put(url, headers=headers, json=payload)
    response.raise_for_status()
    print(f"✅ Αποθηκεύτηκε: https://github.com/{repo}/blob/{branch}/{path}")
    return response.json()


save_df_to_github(
    newsit_2026_df,
    repo="annatsamoyra-prog/data-story-",
    path="newsit_gr_texnologia_2026.csv",
    token=userdata.get("newtoken"),
)


✅ Αποθηκεύτηκε: https://github.com/annatsamoyra-prog/data-story-/blob/main/newsit_gr_texnologia_2026.csv


{'content': {'name': 'newsit_gr_texnologia_2026.csv',
  'path': 'newsit_gr_texnologia_2026.csv',
  'sha': '5e1439844fbe1be8ed5ba3182e2276d98a7310f4',
  'size': 115018,
  'url': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/contents/newsit_gr_texnologia_2026.csv?ref=main',
  'html_url': 'https://github.com/annatsamoyra-prog/data-story-/blob/main/newsit_gr_texnologia_2026.csv',
  'git_url': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/git/blobs/5e1439844fbe1be8ed5ba3182e2276d98a7310f4',
  'download_url': 'https://raw.githubusercontent.com/annatsamoyra-prog/data-story-/main/newsit_gr_texnologia_2026.csv',
  'type': 'file',
  '_links': {'self': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/contents/newsit_gr_texnologia_2026.csv?ref=main',
   'git': 'https://api.github.com/repos/annatsamoyra-prog/data-story-/git/blobs/5e1439844fbe1be8ed5ba3182e2276d98a7310f4',
   'html': 'https://github.com/annatsamoyra-prog/data-story-/blob/main/newsit_gr_texnolog

## Limitations

- Το teaser-block extraction χρησιμοποιεί `doc.find_all("article")` σαν πρώτη επιλογή (τυπικό
  για WordPress), με fallback σε `div` classes -- αν το πρώτο τρέξιμο επιστρέψει 0 teasers,
  έλεγξε το raw HTML μιας σελίδας κατηγορίας στο browser inspector για το πραγματικό container.
- Το boilerplate STOP_HEADINGS/JUNK_LINE_PREFIXES επιβεβαιώθηκε σε **ένα** άρθρο (video-interview
  τύπου, μικρό σε έκταση) -- δες τα πρώτα δείγματα `full_text` του τελικού dataset για τυχόν
  εκπλήξεις σε πιο μεγάλα/διαφορετικού τύπου άρθρα (π.χ. άρθρα με πολλά «Αν τα χάσατε» modules).
- Το `end_page = 40` είναι ανώτατο όριο ασφαλείας· αν το `Min datetime` στο τελικό check δεν
  φτάνει στην αρχή του 2026, χρειάζεται μεγαλύτερο όριο.
